## 1.3.1a Proporción de la población cubierta por sistemas de protección social

Bases:

"datos/1.3.1.A_sh_es.xls"

Cumplir al menos una:
* Prestaciones de ley (derecho a la salud, protección de los medios de subsistencia, servicios sociales necesarios para el bienestar
individual y colectivo, pensión garantizada por el Estado)
* Tienes IMSS o algún Seguro de Salud por parte de tu trabajo o lo adquiriste voluntariamente
* Tienes algún familiar con acceso a la Seguridad Social y te beneficia directamente (por ejemplo, a los hijos menores de 15 años les corresponde atención médica si sus papás cuentan con IMSS)
* Eres jubilado o pensionado
* Tienes 65 años o más y cuentas con pensión de algún programa social, y este ingreso supera la línea de pobreza extrema


In [ ]:
import plotly.graph_objects as go
import pandas as pd
import plotly.express as px
import json
import requests
data_path = "../datos/"

: 

In [ ]:
df = pd.read_excel(data_path + "1.3.1.A_sh_es.xls", sheet_name=0, header = None)
df = df.iloc[7:-10] # Eliminar todos los renglones de texto
df.columns = df.iloc[0] # El encabezado es el primer renglón
df = df[1:] # Eliminar el primer renglon (ya es el encazabezado)
df.columns = ["cvegeo","Entidad", "2022", "2020", "2018", "2016"]

# Cambiar los nombres del df para que coincidan con los del json
df.iloc[4, 1] = "Coahuila"
df.iloc[15,1] = "Michoacán"
df.iloc[29,1] = "Veracruz"

df_plot = df.melt(
    id_vars=['Entidad'],
    value_vars=['2016', '2018', '2020', '2022'],
    var_name='Año',
    value_name='Porcentaje'
)

# 3. Crear la gráfica de líneas
fig = px.line(
    df_plot,
    x='Año',
    y='Porcentaje',
    color='Entidad', # Esto crea una serie (línea) por cada categoría
    title='Proporción de la población cubierta por sistemas protección social,<br>'+ 'del año 2016 al 2022, por entidad federativa',
    markers=True # Para que se vean los puntos en cada año
)

# Mejorar el diseño
fig.update_layout(xaxis_type='category') # Para que los años se traten como etiquetas
fig.show()

In [ ]:
# --- 1. Obtener el GeoJSON de México (Estados) ---
repo_url = 'https://raw.githubusercontent.com/angelnmara/geojson/master/mexicoHigh.json'
repo_url_municps = 'https://raw.githubusercontent.com/angelnmara/geojson/master/MunicipiosMexico.json'
mx_states_geojson = requests.get(repo_url).json()


# Partición del porcentaje en categorías
df_plot['Categoría'] = pd.cut(
    df_plot['Porcentaje'],
    bins=[0, 25, 50, 100],
    labels=["[0,25]", "(25,50]", "(50,100]"],
    include_lowest=True,
    right=True
)

colores = {
    "[0,25]": "#d14e39",
    "(25,50]": "#efd720",
    "(50,100]": "#87aa16"
}

fig = px.choropleth(
    df_plot,
    geojson=mx_states_geojson,
    locations='Entidad',
    featureidkey='properties.name',
    color='Categoría',             # Coloreamos por la columna de texto
    animation_frame='Año',
    color_discrete_map=colores,    # Forzamos los colores definidos arriba
    hover_name='Entidad',
    hover_data={'Porcentaje':True, 'Entidad':False} # Mostramos el número real en el hover
)

# 4. Ajustar Proyección y Enfoque
fig.update_geos(
    fitbounds="locations",
    visible=False
)

fig.update_layout(
    title_text='Proporción de la población cubierta por sistemas protección social<br><sup>Animación del 2016 al 2022</sup>',
    margin={"r":0,"t":80,"l":0,"b":0}
)

fig.show()

Respecto de las prestaciones de ley...
## 1n.3.2.a Porcentaje de población ocupada con hijos(as) de 6 años o menos que tiene acceso a **guarderías** como prestación laboral, por desglose geográfico

Bases:

"datos/1N.3.2.A_sh_es.xls"


In [ ]:
df = pd.read_excel(data_path+"1N.3.2.A_sh_es.xls", sheet_name=0, header = None)
df = df.iloc[7:-16] # Eliminar todos los renglones de texto
df.columns = df.iloc[0] # El encabezado es el primer renglón
df = df[1:] # Eliminar el primer renglon (ya es el encazabezado)
df.columns = ["cvegeo","Entidad", "2022", "2020", "2018", "2016"]

anios = ['2022','2020','2018','2016']
liminfs = ['inf2022','inf2020','inf2018','inf2016']
limsups = ['sup2022','sup2020','sup2018','sup2016']
sd = ['sd2022','sd2020','sd2018','sd2016']
mapa_notas = {'a/': 0.15, 'b/': 0.25, 'c/':0.2}

for anio in anios:
  extraer = df[anio].str.extract(r'(?P<Numero>[0-9.]+)\s*(?P<Sufijo>.*)')
  sd = ['sd2022','sd2020','sd2018','sd2016']
  df[sd[anios.index(anio)]] = extraer['Sufijo']
  df[sd[anios.index(anio)]] = df[sd[anios.index(anio)]].map(mapa_notas)
  df[anio] = pd.to_numeric(extraer['Numero'])
  df[sd[anios.index(anio)]] = df[anio]*df[sd[anios.index(anio)]]
  df[liminfs[anios.index(anio)]] = df[anio] - df[sd[anios.index(anio)]]
  df[limsups[anios.index(anio)]] = df[anio] + df[sd[anios.index(anio)]]

df.iloc[4, 1] = "Coahuila"
df.iloc[15,1] = "Michoacán"
df.iloc[29,1] = "Veracruz"

lista_estados = df['Entidad'].unique()
anios = ['2016', '2018', '2020', '2022']

fig = go.Figure()

# --- Paso A: Añadir Trazas (3 por cada estado) ---
for estado in lista_estados:
    df_e = df[df['Entidad'] == estado]

    # Extraer valores para los ejes
    y_vals = [df_e[f'{a}'].values[0] for a in anios]
    y_upper = [df_e[f'sup{a}'].values[0] for a in anios]
    y_lower = [df_e[f'inf{a}'].values[0] for a in anios]

    # 1. Traza Upper (Invisible)
    fig.add_trace(go.Scatter(
        x=anios, y=y_upper, mode='lines',
        line=dict(width=0), showlegend=False, visible=False
    ))

    # 2. Traza Lower (Sombreado)
    fig.add_trace(go.Scatter(
        x=anios, y=y_lower, mode='lines',
        line=dict(width=0), fill='tonexty',
        fillcolor='rgba(191, 220, 139, 0.5)',
        showlegend=False, visible=False
    ))

    # 3. Traza Principal (Línea)
    fig.add_trace(go.Scatter(
        x=anios, y=y_vals, mode='lines+markers',
        name=estado, line=dict(color='#457d36'),
        visible=False
    ))

# --- Paso B: Activar el primer estado (las primeras 3 trazas) ---
for j in range(3):
    fig.data[j].visible = True

# --- Paso C: Crear botones del Menú ---
botones = []
num_trazas_por_estado = 3

for i, estado in enumerate(lista_estados):
    # Crear máscara de visibilidad
    # Si hay 32 estados, habrá 32 * 3 = 96 trazas en total
    visibilidad = [False] * len(fig.data)

    # Activar el bloque de 3 trazas que corresponden a este estado
    inicio = i * num_trazas_por_estado
    visibilidad[inicio : inicio + num_trazas_por_estado] = [True, True, True]

    botones.append(dict(
        label=estado,
        method="update",
        args=[{"visible": visibilidad},
              {"title": f"Población con acceso a Guarderías en {estado}"}]
    ))

# --- Paso D: Configurar Layout ---
fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=botones,
        x=0.1, y=1.2,
        xanchor="left", yanchor="top"
    )],
    xaxis_title="Año",
    yaxis_title="Valor / Proporción",
    hovermode="x unified"
)

fig.show()

In [ ]:
df_plot = df.melt(
    id_vars=['Entidad'], # renglones
    value_vars=['2016', '2018', '2020', '2022'],
    var_name='Año',
    value_name='Porcentaje'
)

# Partición del porcentaje en categorías
df_plot['Categoría'] = pd.cut(
    df_plot['Porcentaje'],
    bins=[0, 15, 30, 100],
    labels=["[0,15]", "(15,30]", "(30,100]"],
    include_lowest=True,
    right=True
)

dummy = pd.DataFrame({'Entidad': ['Dummy'], 'Año': ['2016'], 'Porcentaje': ['31%'], 'Categoría': ['(30,100]']})
df_plot = pd.concat([dummy,df_plot], ignore_index=True)

colores = {
    "[0,15]": "#d14e39",
    "(15,30]": "#efd720",
    "(30,100]": "#87aa16"
}

fig = px.choropleth(
    df_plot,
    geojson=mx_states_geojson,
    locations='Entidad',
    featureidkey='properties.name',
    color='Categoría',
    animation_frame='Año',
    color_discrete_map=colores,
    hover_name='Entidad',
    hover_data={'Porcentaje':True, 'Entidad':False}
)

fig.update_geos(
    fitbounds="locations",
    visible=False
)

fig.update_layout(
    title_text='Proporción de la población ocupada con acceso a guardería<br><sup>Animación del 2016 al 2022</sup>',
    margin={"r":0,"t":80,"l":0,"b":0}
)

fig.show()


## 1n.3.1.a  Porcentaje de población de **65 años** o más que recibe jubilación o **pensión** (contributiva o no contributiva) por un monto igual o mayor al valor promedio de la línea de pobreza por ingresos, por desglose geográfico

Bases:

"datos/1N.3.1.A_sh_es.xls"

In [ ]:
df = pd.read_excel(data_path+ "1N.3.1.A_sh_es.xls", sheet_name=0, header = None)
df = df.iloc[7:-19] # Eliminar todos los renglones de texto
df.columns = df.iloc[0] # El encabezado es el primer renglón
df = df[1:] # Eliminar el primer renglon (ya es el encazabezado)
df.columns = ["cvegeo","Entidad", "2022", "2020", "2018", "2016"]

anios = ['2022','2020','2018','2016']
liminfs = ['inf2022','inf2020','inf2018','inf2016']
limsups = ['sup2022','sup2020','sup2018','sup2016']
sd = ['sd2022','sd2020','sd2018','sd2016']
mapa_notas = {'a/': 0.15, 'b/': 0.25, 'c/':0.2}

for anio in anios:
  extraer = df[anio].str.extract(r'(?P<Numero>[0-9.]+)\s*(?P<Sufijo>.*)')
  sd = ['sd2022','sd2020','sd2018','sd2016']
  df[sd[anios.index(anio)]] = extraer['Sufijo']
  df[sd[anios.index(anio)]] = df[sd[anios.index(anio)]].map(mapa_notas)
  df[anio] = pd.to_numeric(extraer['Numero'])
  df[sd[anios.index(anio)]] = df[anio]*df[sd[anios.index(anio)]]
  df[liminfs[anios.index(anio)]] = df[anio] - df[sd[anios.index(anio)]]
  df[limsups[anios.index(anio)]] = df[anio] + df[sd[anios.index(anio)]]

df.iloc[4, 1] = "Coahuila"
df.iloc[15,1] = "Michoacán"
df.iloc[29,1] = "Veracruz"

lista_estados = df['Entidad'].unique()
anios = ['2016', '2018', '2020', '2022']

fig = go.Figure()

# --- Paso A: Añadir Trazas (3 por cada estado) ---
for estado in lista_estados:
    df_e = df[df['Entidad'] == estado]

    # Extraer valores para los ejes
    y_vals = [df_e[f'{a}'].values[0] for a in anios]
    y_upper = [df_e[f'sup{a}'].values[0] for a in anios]
    y_lower = [df_e[f'inf{a}'].values[0] for a in anios]

    # 1. Traza Upper (Invisible)
    fig.add_trace(go.Scatter(
        x=anios, y=y_upper, mode='lines',
        line=dict(width=0), showlegend=False, visible=False
    ))

    # 2. Traza Lower (Sombreado)
    fig.add_trace(go.Scatter(
        x=anios, y=y_lower, mode='lines',
        line=dict(width=0), fill='tonexty',
        fillcolor='rgba(191, 220, 139, 0.5)',
        showlegend=False, visible=False
    ))

    # 3. Traza Principal (Línea)
    fig.add_trace(go.Scatter(
        x=anios, y=y_vals, mode='lines+markers',
        name=estado, line=dict(color='rgba(191, 220, 139, 0.5)'),
        visible=False
    ))

# --- Paso B: Activar el primer estado (las primeras 3 trazas) ---
for j in range(3):
    fig.data[j].visible = True

# --- Paso C: Crear botones del Menú ---
botones = []
num_trazas_por_estado = 3

for i, estado in enumerate(lista_estados):
    # Crear máscara de visibilidad
    # Si hay 32 estados, habrá 32 * 3 = 96 trazas en total
    visibilidad = [False] * len(fig.data)

    # Activar el bloque de 3 trazas que corresponden a este estado
    inicio = i * num_trazas_por_estado
    visibilidad[inicio : inicio + num_trazas_por_estado] = [True, True, True]

    botones.append(dict(
        label=estado,
        method="update",
        args=[{"visible": visibilidad},
              {"title": f"Serie de Tiempo con IC: {estado}"}]
    ))

# --- Paso D: Configurar Layout ---
fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=botones,
        x=0.1, y=1.2,
        xanchor="left", yanchor="top"
    )],
    xaxis_title="Año",
    yaxis_title="Valor / Proporción",
    hovermode="x unified"
)

fig.show()

In [ ]:
df_plot = df.melt(
    id_vars=['Entidad'], # renglones
    value_vars=['2016', '2018', '2020', '2022'],
    var_name='Año',
    value_name='Porcentaje'
)

# Partición del porcentaje en categorías
df_plot['Categoría'] = pd.cut(
    df_plot['Porcentaje'],
    bins=[0, 25, 50, 100],
    labels=["[0,25]", "(25,50]", "(50,100]"],
    include_lowest=True,
    right=True
)


colores = {
    "[0,25]": "#d14e39",
    "(25,50]": "#efd720",
    "(50,100]": "#87aa16"
}


fig = px.choropleth(
    df_plot,
    geojson=mx_states_geojson,
    locations='Entidad',
    featureidkey='properties.name',
    color='Categoría',
    animation_frame='Año',
    color_discrete_map=colores,
    hover_name='Entidad',
    hover_data={'Porcentaje':True, 'Entidad':False}
)

fig.update_geos(
    fitbounds="locations",
    visible=False
)

fig.update_layout(
    title_text='Proporción de la población mayor de 65 años con pensión digna<br><sup>Animación del 2016 al 2022</sup>',
    margin={"r":0,"t":80,"l":0,"b":0}
)

fig.show()

## 1.4.1  Proporción de la población que vive en hogares con acceso a los servicios básicos

Bases:

"datos/1.4.1_sh_es.xls"


In [ ]:
df = pd.read_excel(data_path+"1.4.1_sh_es.xls", sheet_name=0, header = None)
df = df.iloc[7:-3] # Eliminar todos los renglones de texto
df.columns = df.iloc[0] # El encabezado es el primer renglón
df = df[1:] # Eliminar el primer renglon (ya es el encazabezado)
df.columns = ["cvegeo","Entidad","2024" ,"2022", "2020", "2018", "2016"]

# Cambiar los nombres del df para que coincidan con los del json
df.iloc[4, 1] = "Coahuila"
df.iloc[15,1] = "Michoacán"
df.iloc[29,1] = "Veracruz"

df_plot = df.melt(
    id_vars=['Entidad'],
    value_vars=['2016', '2018', '2020', '2022', '2024'],
    var_name='Año',
    value_name='Porcentaje'
)

# 3. Crear la gráfica de líneas
fig = px.line(
    df_plot,
    x='Año',
    y='Porcentaje',
    color='Entidad', # Esto crea una serie (línea) por cada categoría
    title='Proporción de la población que vive en hogares con acceso a servicios básicos,<br>'+ 'del año 2016 al 2022, por entidad federativa',
    markers=True # Para que se vean los puntos en cada año
)

# Mejorar el diseño
fig.update_layout(xaxis_type='category') # Para que los años se traten como etiquetas
fig.show()

In [ ]:
# --- 1. Obtener el GeoJSON de México (Estados) ---
repo_url = 'https://raw.githubusercontent.com/angelnmara/geojson/master/mexicoHigh.json'
repo_url_municps = 'https://raw.githubusercontent.com/angelnmara/geojson/master/MunicipiosMexico.json'
mx_states_geojson = requests.get(repo_url).json()


# Partición del porcentaje en categorías
df_plot['Categoría'] = pd.cut(
    df_plot['Porcentaje'],
    bins=[0, 50, 75, 100],
    labels=["[0,50]", "(50,75]", "(75,100]"],
    include_lowest=True,
    right=True
)

df_plot = df_plot.sort_values(by=['Año', 'Entidad'])

colores = {
    "[0,50]": "#d14e39",
    "(50,75]": "#efd720",
    "(75,100]": "#87aa16"
}

fig = px.choropleth(
    df_plot,
    geojson=mx_states_geojson,
    locations='Entidad',
    featureidkey='properties.name',
    color='Categoría',
    animation_frame='Año',
    category_orders={
        "Categoría": ["[0,50]", "(50,75]", "(75,100]"]},
    color_discrete_map=colores,
    hover_name='Entidad',
    hover_data={'Porcentaje':True, 'Entidad':False}
)

# 4. Ajustar Proyección y Enfoque
fig.update_geos(
    fitbounds="locations",
    visible=False
)

fig.update_layout(
    title_text='Proporción de la población que vive en hogares con acceso a servicios básicos<br><sup>Animación del 2016 al 2024</sup>',
    margin={"r":0,"t":80,"l":0,"b":0}
)

fig.show()